In [ ]:
import re, subprocess


# -- Parameters ----------------------------------------
MAX_CAP_CHARS = 2000   # Maximum number of caption characters
MAX_SENTENCES = 6      # Maximum number of sentences


# -- Pattern Definitions ---------------------------------
CAP_START = re.compile(
    r'^\s*(?:Figure|Fig\.?)\s+(\d+[a-zA-Z]?)\s*(?:[:.|\s]\s*)(.*)',
    re.IGNORECASE
)


STOP_HARD = re.compile(
    r'^\s*(?:Figure|Fig\.?|Table|Tbl\.?|Algorithm|Appendix|'
    r'References|Acknowledgment|Acknowledgement|Supplementary)',
    re.IGNORECASE
)


# Caption continuation pattern: continue collecting caption even if lowercase
CAPTION_CONT = re.compile(
    r'^\s*(?:'
    r'\([a-zA-Z0-9]\)|'
    r'(?:Left|Right|Top|Bottom|Center|Middle|Upper|Lower)\s*[:(]|'
    r'(?:Solid|Dashed|Dotted|Colored?)\s*:|'
    r'(?:Panel|Row|Column|Layer)\s*[:(]|'
    r'(?:For each|Each|Per|From left|From right)\s|'
    r'(?:Red|Blue|Green|Orange|Purple|Yellow|Black|White|'
    r'    Gray|Grey|Dark|Light)\s*:|'
    r'(?:Arrows?|Bars?|Lines?|Circles?|Points?|Dots?|Boxes?)\s*:'
    r')',
    re.IGNORECASE
)


# Body transition expression: caption ends when a complete sentence is followed by this
BODY_TRANSITION = re.compile(
    r'^(?:'
    r'In this (?:paper|work|section|study)|'
    r'We (?:address|present|propose|study|train|use|show|find|note|'
    r'    evaluate|follow|extend|further|utilize|compare|examine|'
    r'    analyze|now|next|also consider|leverage|describe|'
    r'    introduce|investigate)|'
    r'Our (?:approach|method|model|main|key|work|goal|'
    r'    contribution|focus|experiments?)|'
    r'This (?:paper|work|section|chapter|approach|model|method)|'
    r'Building on|Following (?:this|prior|previous|the)|'
    r'To (?:this end|evaluate|validate|address|test|better|further)|'
    r'Moreover[,\s]|Furthermore[,\s]|Note that|'
    r'Specifically[,\s]|Here we (?:present|propose|show|study|address)|'
    r'As (?:a result|shown in|described in|discussed in|mentioned)|'
    r'Unlike |While |Although |Despite |However[,\s]'
    r')',
    re.IGNORECASE
)


# -- Utility Functions -----------------------------------
def count_sentences(text):
    """Estimate sentence count by period, exclamation mark, question mark + next uppercase pattern"""
    t = re.sub(r'(?:e\.g\.|i\.e\.|Fig\.|Eq\.|Sec\.|et al\.|cf\.|vs\.)',
               '___', text)
    return len(re.findall(r'[.!?]["\’"]?\s*(?=[A-Z]|\Z)', t))


def ends_with_sentence(text):
    t = text.rstrip()
    return bool(t) and t[-1] in '.!?'


def starts_lowercase(line):
    """Lowercase start → candidate for two-column typesetting artifact. Exclude starts with '('"""
    return bool(line) and line[0].islower() and not line.startswith('(')


# -- Main Extraction Function ----------------------------
def extract_captions(pdf_path):
    result = subprocess.run(
        ['pdftotext', '-enc', 'UTF-8', pdf_path, '-'],
        capture_output=True, text=True, timeout=60, errors='replace'
    )
    lines    = result.stdout.split('\n')
    captions = {}
    i = 0
    while i < len(lines):
        m = CAP_START.match(lines[i])
        if not m:
            i += 1
            continue


        fig_id   = m.group(1)            # e.g.: "2", "3a"
        cap_text = m.group(2).strip()    # First line of caption
        i += 1


        while i < len(lines):
            stripped = lines[i].strip()


            # -- Termination Conditions ---------------------
            if not stripped:              break   # Empty line
            if STOP_HARD.match(lines[i]): break   # Structural label
            if len(cap_text) >= MAX_CAP_CHARS: break
            if count_sentences(cap_text) >= MAX_SENTENCES: break


            ews = ends_with_sentence(cap_text)
            cc  = bool(CAPTION_CONT.match(stripped))


            if ews and not cc:
                # Lowercase start → two-column typesetting artifact
                if starts_lowercase(stripped): break
                # Detect body transition expression
                if BODY_TRANSITION.match(stripped): break


            cap_text += ' ' + stripped
            i += 1


        captions[f'Figure {fig_id}'] = cap_text.strip()


    return captions